Zillow Housing Data

In [1]:
import pandas as pd
from google.colab import files

Zillow_data = pd.read_csv("ZillowHousingData.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'ZillowHousingData.csv'

Geospatial Data and Density Maps (Yelp API)

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load the Yelp output
df = pd.read_csv("yelp_businesses.csv")

# Filter to quality listings only (proxy for established businesses)
df = df[df["review_count"] > 5]

# Load tract shapefile to get area in square miles
tracts = gpd.read_file("bay_area_tracts.shp")
tracts.columns = [c.replace("00","").replace("10","") for c in tracts.columns]
if "CTIDFP" in tracts.columns:
    tracts = tracts.rename(columns={"CTIDFP": "GEOID"})

# Compute tract area in square miles
tracts = tracts.to_crs("EPSG:3310")  # California Albers (meters)
tracts["area_sqmi"] = tracts.geometry.area / 2_589_988  # convert m² to sq miles

# Count businesses per tract per category
density = (
    df.groupby(["actual_geoid", "matched_category"])
    .size()
    .reset_index(name="count")
)

# Pivot so each category becomes a column
density_wide = density.pivot(index="actual_geoid", columns="matched_category", values="count").fillna(0)
density_wide.columns = [f"yelp_{col}_count" for col in density_wide.columns]
density_wide = density_wide.reset_index()

density_wide["actual_geoid"] = density_wide["actual_geoid"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(11)
tracts["GEOID"] = tracts["GEOID"].astype(str).str.zfill(11)

# Merge with tract areas and normalize to per square mile
density_wide = density_wide.merge(
    tracts[["GEOID", "area_sqmi"]], left_on="actual_geoid", right_on="GEOID", how="left"
)
for col in [c for c in density_wide.columns if c.startswith("yelp_")]:
    density_wide[col.replace("_count", "_density")] = density_wide[col] / density_wide["area_sqmi"]

# Save the feature table
density_wide.to_csv("yelp_features.csv", index=False)
print(f"Saved {len(density_wide)} tracts to yelp_features.csv")

In [ ]:
files.download('yelp_features.csv')

ACS Tract-Level Data

In [2]:
!pip install censusdata
import censusdata
import pandas as pd
import numpy as np

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 58.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for censusdata: filename=CensusData-1.15.post1-py3-none-any.whl size=28205744 sha256=885856756366be9447c77ff60b50a8e397d70104c7cdfac796d78090ebf20a28
  Stored in directory: /root/.cache/pip/wheels/54/5e/eb/518ccd7738e6b9b35d9fb3d226d45979066ec367ed26ad1369
Successfully built censusdata


In [3]:
counties = {
    "San_Francisco": "075",
    "Alameda_Oakland": "001",
    "Santa_Clara_San_Jose": "085"
}

In [4]:
def get_acs(year):

    # ✅ Updated variable list (removed problematic education variable)
    vars = [
        'B19013_001E',  # median household income
        'B25064_001E',  # median gross rent
        'B01003_001E'   # total population
    ]

    frames = []

    for name, county in counties.items():

        df = censusdata.download(
            'acs5',
            year,
            censusdata.censusgeo([
                ('state', '06'),
                ('county', county),
                ('tract', '*')
            ]),
            vars
        )

        df = df.reset_index()
        df['year'] = year
        df['county'] = name

        frames.append(df)

    return pd.concat(frames, ignore_index=True)

In [5]:
def clean_acs(df):

    df.columns = [
        'geo', 'income', 'rent', 'population',
        'year', 'county'
    ]

    df['tract_id'] = df['geo'].astype(str)

    df = df.drop(columns=['geo'])

    return df

In [6]:
df_2010 = clean_acs(get_acs(2010))
df_2015 = clean_acs(get_acs(2015))
df_2020 = clean_acs(get_acs(2020))

In [7]:
# Combine all cleaned ACS dataframes into one
combined_acs_df = pd.concat([df_2010, df_2015, df_2020], ignore_index=True)

print("Combined DataFrame shape:", combined_acs_df.shape)
display(combined_acs_df.head())

Combined DataFrame shape: (2891, 6)


,income,rent,population,year,county,tract_id
0,54095.0,1318.0,3744,2010,San_Francisco,"Census Tract 101, San Francisco County, Califo..."
1,89096.0,1554.0,4184,2010,San_Francisco,"Census Tract 102, San Francisco County, Califo..."
2,99840.0,1995.0,4285,2010,San_Francisco,"Census Tract 103, San Francisco County, Califo..."
3,77857.0,1780.0,4154,2010,San_Francisco,"Census Tract 104, San Francisco County, Califo..."
4,117115.0,2001.0,2429,2010,San_Francisco,"Census Tract 105, San Francisco County, Califo..."


In [8]:
# Export to CSV
csv_filename = 'combined_acs_data.csv'
combined_acs_df.to_csv(csv_filename, index=False)
print(f"Data exported to {csv_filename}")

# Download the CSV file
from google.colab import files
files.download(csv_filename)

Data exported to combined_acs_data.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# Export to Excel
excel_filename = 'combined_acs_data.xlsx'
combined_acs_df.to_excel(excel_filename, index=False)
print(f"Data exported to {excel_filename}")

# Download the Excel file
from google.colab import files
files.download(excel_filename)

Data exported to combined_acs_data.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>